In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb
import joblib

In [7]:
# Load the dataset using a raw string for the file path
data = pd.read_csv(r"C:\Users\rohit\OneDrive\Desktop\water\ip\water_potability copy.csv")

In [8]:
# Check for missing values
print("Missing values (counts):\n", data.isnull().sum())

# Fill missing values with class-specific means
data.loc[data["Potability"] == 0] = data[data["Potability"] == 0].fillna(data[data["Potability"] == 0].mean())
data.loc[data["Potability"] == 1] = data[data["Potability"] == 1].fillna(data[data["Potability"] == 1].mean())

# Verify missing values are handled
print("Missing values after cleaning:\n", data.isnull().sum())

Missing values (counts):
 ph                 491
Hardness             0
Solids               0
Chloramines          0
Sulfate            781
Conductivity         0
Organic_carbon       0
Trihalomethanes    162
Turbidity            0
Potability           0
dtype: int64
Missing values after cleaning:
 ph                 0
Hardness           0
Solids             0
Chloramines        0
Sulfate            0
Conductivity       0
Organic_carbon     0
Trihalomethanes    0
Turbidity          0
Potability         0
dtype: int64


In [9]:
# Clip pH and create normalized pH feature
data["ph"] = data["ph"].clip(5, 9)
data["ph_normalized"] = (data["ph"] - 5) / (9 - 5) * 0.1  # Scaled down

# Clip Sulfate
data["Sulfate"] = data["Sulfate"].clip(100, 300)

# Handle infinite values and drop NaNs
data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.dropna(inplace=True)

In [10]:
# Set random seed for reproducibility
np.random.seed(42)
n_synthetic = 50

# Synthetic bad cases: Neutral pH, high other features
synthetic_bad = pd.DataFrame({
    "ph": np.random.uniform(6.5, 8.5, n_synthetic),
    "Solids": np.random.uniform(12000, 25000, n_synthetic),
    "Chloramines": np.random.uniform(6, 10, n_synthetic),
    "Sulfate": np.random.uniform(200, 300, n_synthetic),
    "Turbidity": np.random.uniform(6, 12, n_synthetic),
    "Potability": [0] * n_synthetic
})

# Synthetic good cases: Neutral pH, low other features
synthetic_good = pd.DataFrame({
    "ph": np.random.uniform(6.5, 8.5, n_synthetic * 4),  # 200 samples for Potable
    "Solids": np.random.uniform(500, 4000, n_synthetic * 4),
    "Chloramines": np.random.uniform(1, 4, n_synthetic * 4),
    "Sulfate": np.random.uniform(100, 200, n_synthetic * 4),
    "Turbidity": np.random.uniform(1, 2.5, n_synthetic * 4),
    "Potability": [1] * n_synthetic * 4
})

# Apply feature engineering to synthetic data
for df in [synthetic_bad, synthetic_good]:
    df["ph"] = df["ph"].clip(5, 9)
    df["ph_normalized"] = (df["ph"] - 5) / (9 - 5) * 0.1
    df["Sulfate"] = df["Sulfate"].clip(100, 300)

# Combine original and synthetic data
data = pd.concat([data, synthetic_bad, synthetic_good], ignore_index=True)

# Check class distribution
print("Class distribution:\n", data["Potability"].value_counts(normalize=True).to_string())

Class distribution:
 Potability
0    0.580828
1    0.419172


In [11]:
# Define features (X) and target (y)
feature_names = ["ph_normalized", "Solids", "Chloramines", "Sulfate", "Turbidity"]
X = data[feature_names]
y = data["Potability"]

In [12]:
# Compute class weights for imbalanced classes
class_weights = {0: len(y) / (2 * (y == 0).sum()), 1: len(y) / (2 * (y == 1).sum())}

# Apply additional weights based on feature thresholds
sample_weight = np.array([
    class_weights[yi] * (
        2.2 if (row["Solids"] > 10000 or row["Chloramines"] > 6 or row["Turbidity"] > 5 or row["Sulfate"] > 250) and yi == 0 else
        2.2 if (row["Solids"] < 5000 and row["Chloramines"] < 4 and row["Turbidity"] < 3 and row["Sulfate"] < 200) and yi == 1 else 1
    ) for yi, (_, row) in zip(y, data.iterrows())
])

In [13]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test, sample_weight_train, _ = train_test_split(
    X, y, sample_weight, test_size=0.2, random_state=42
)

In [14]:
# Initialize and train XGBoost model
print("Training XGBoost...")
model_xgb = xgb.XGBClassifier(
    n_estimators=450,
    learning_rate=0.02,
    max_depth=4,
    min_child_weight=2,
    subsample=0.9,
    colsample_bytree=0.7,
    random_state=42,
    eval_metric="logloss"
)
model_xgb.fit(X_train, y_train, sample_weight=sample_weight_train)

Training XGBoost...


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.02, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=2, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=450, n_jobs=None,
              num_parallel_tree=None, ...)

In [15]:
# Calculate training and testing accuracy
train_score_xgb = model_xgb.score(X_train, y_train)
test_score_xgb = model_xgb.score(X_test, y_test)
print(f"XGBoost Training Accuracy: {train_score_xgb:.4f}")
print(f"XGBoost Testing Accuracy: {test_score_xgb:.4f}")

# Print feature importance
print("\nXGBoost Feature Importance:")
for name, imp in zip(feature_names, model_xgb.feature_importances_):
    print(f"{name}: {imp:.4f}")

XGBoost Training Accuracy: 0.7337
XGBoost Testing Accuracy: 0.7280

XGBoost Feature Importance:
ph_normalized: 0.2667
Solids: 0.1443
Chloramines: 0.1264
Sulfate: 0.3959
Turbidity: 0.0667


In [16]:
# Create sample inputs for prediction
good_input = pd.DataFrame([[7.2, 500, 2.5, 200, 1.5]], columns=feature_names)
bad_input = pd.DataFrame([[4.5, 15000, 7.0, 300, 8.0]], columns=feature_names)

# Normalize pH for sample inputs
good_input["ph_normalized"] = (good_input["ph_normalized"] - 5) / (9 - 5) * 0.1
bad_input["ph_normalized"] = (bad_input["ph_normalized"] - 5) / (9 - 5) * 0.1

# Make predictions
print("\nSample Predictions:")
print(f"Good Water: {'Potable' if model_xgb.predict(good_input)[0] == 1 else 'Not Potable'}")
print(f"Bad Water: {'Potable' if model_xgb.predict(bad_input)[0] == 1 else 'Not Potable'}")


Sample Predictions:
Good Water: Potable
Bad Water: Not Potable


In [20]:
# Import required metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Make predictions
y_train_pred = model_xgb.predict(X_train)
y_test_pred = model_xgb.predict(X_test)

# Calculate metrics
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)

# Print metrics
print("\nXGBoost Performance Metrics:")
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

# Print feature importance
print("\nXGBoost Feature Importance:")
for name, imp in zip(feature_names, model_xgb.feature_importances_):
    print(f"{name}: {imp:.4f}")

# Store confusion matrix for visualization
cm = confusion_matrix(y_test, y_test_pred)


XGBoost Performance Metrics:
Training Accuracy: 0.7337
Testing Accuracy: 0.7280
Precision: 0.9153
Recall: 0.3724
F1-Score: 0.5294

XGBoost Feature Importance:
ph_normalized: 0.2667
Solids: 0.1443
Chloramines: 0.1264
Sulfate: 0.3959
Turbidity: 0.0667


In [25]:
# Import visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Feature Importance Plot
plt.figure(figsize=(10, 6))
sns.barplot(x=model_xgb.feature_importances_, y=feature_names)
plt.title("XGBoost Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "xgboost_feature_importance.png"))
plt.close()

# Confusion Matrix Plot
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("XGBoost Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "xgboost_confusion_matrix.png"))
plt.close()

print(f"\nVisualizations saved to {output_dir}/")

ModuleNotFoundError: No module named 'matplotlib'